In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté
# On fusionne les deux types de données du Bronze (BATCH et STREAMING)

# ------------------------------------------------------------
# Configuration UC
# ------------------------------------------------------------

spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA gold")

silver_stream_table = "main.silver.transactions_silver_stream"

print(f"Lecture Silver Streaming depuis : {silver_stream_table}")

# ------------------------------------------------------------
# Lecture en streaming de la table Silver
# ------------------------------------------------------------
silver_stream = spark.readStream.table(silver_stream_table)

# ------------------------------------------------------------
# KPI globaux en temps réel
# ------------------------------------------------------------

kpi_checkpoint = (
    "/Volumes/main/gold/gold_volume/"
    "_checkpoints/kpi_stream"
)

df_kpi_stream = (
    silver_stream
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("amount").alias("total_amount"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum(F.when(F.col("is_fraud") == 1, F.col("amount")).otherwise(0)).alias("fraud_amount"),  # noqa: E501
        F.sum(F.when(F.col("is_fraud") == 0, F.col("amount")).otherwise(0)).alias("legit_amount"),  # noqa: E501
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

query_kpi = (
    df_kpi_stream.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", kpi_checkpoint)
    .trigger(once=True)
    .table("fraud_kpi_gold")
)

print("✓ KPI Gold Streaming démarré → main.gold.fraud_kpi_gold")

# Enrichissement Time en miutes / heures /jour
df_gold_stream = (
    silver_stream
    .withColumn("minute", (F.col("time") / 60).cast("int"))
    .withColumn("hour", (F.col("time") / 3600).cast("int"))
    .withColumn("day", (F.col("time") / 86400).cast("int"))
)

# Fraude par minute (streaming)
minute_checkpoint = (
    "/Volumes/main/gold/gold_volume/"
    "_checkpoints/minute_stream"
)
df_fraud_by_minute_stream  = (
    df_gold_stream
    .groupBy("minute")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

query_minute = (
    df_fraud_by_minute_stream.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", minute_checkpoint)
    .trigger(once=True)
    .table("fraud_by_minute_gold")
)

print("✓ Fraude par minute Streaming → main.gold.fraud_by_minute_gold")

# Fraude par heure (streaming)
hour_checkpoint = (
    "/Volumes/main/gold/gold_volume/"
    "_checkpoints/hour_stream"
)

df_fraud_by_hour_stream = (
    df_gold_stream
    .groupBy("hour")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

query_hour = (
    df_fraud_by_hour_stream.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", hour_checkpoint)
    .trigger(once=True)
    .table("fraud_by_hour_gold")
)

print("✓ Fraude par heure Streaming → main.gold.fraud_by_hour_gold")


# Fraude par jour (streaming)
day_checkpoint = (
    "/Volumes/main/gold/gold_volume/"
    "_checkpoints/day_stream"
)

df_fraud_by_day_stream = (
    df_gold_stream
    .groupBy("day")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

query_day = (
    df_fraud_by_day_stream.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", day_checkpoint)
    .trigger(once=True)
    .table("fraud_by_day_gold")
)

print("✓ Fraude par jour Streaming → main.gold.fraud_by_day_gold")
print("✓ Streaming GOLD opérationnel — KPI temps réel mis à jour en continu.")